# Croatian category classifier — XLM-RoBERTa on translated PatternReframe

Stage 4 of the thesis. Mirrors `02_03_roberta_patternReframe_soft_alpha.ipynb` (Chapter 4.4.4–4.4.5) but with:
- **Model**: `xlm-roberta-base` (multilingual; supports Croatian). Optionally `classla/bcms-bertic` for the South-Slavic-specialist comparison run.
- **Input column**: `thought_hr` (Croatian translation produced in Phase D).
- **Label column**: `pattern` and the `marked_*` columns. Both are language-agnostic — the English category labels apply directly to the Croatian translations.
- **Output names**: suffixed with `_hr_<model_tag>` so the artifacts never collide with the English run.
- **α = 0.3** to match the production setting selected in Chapter 4.4.5 (best weighted F1 across the four α ∈ {1.0, 0.5, 0.3, 0.0} evaluated in the English run).

All other hyperparameters (learning rate, weight decay, epochs, batch size, warmup, max length, seed, early stopping patience) match the English run exactly. Any difference in test-set performance is therefore attributable to the language switch or to translation quality, not to a configuration drift.

In [ ]:
# === Cell 1: Setup ===
!pip install -q transformers datasets accelerate evaluate

# === Cell 2: Mount Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === Cell 3: Imports + GPU check ===
import os, time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback,
)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device:         {torch.cuda.get_device_name(0)}")
print(f"GPU memory:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# === Cell 4: Load Croatian bilingual splits and rebuild label dicts ===
DATA_DIR = "/content/drive/MyDrive/thesis_stage2"

# === Model choice ===
# Primary run:    "xlm-roberta-base"
# Comparison run: "classla/bcms-bertic"  — South-Slavic specialist
MODEL_NAME = "xlm-roberta-base"
MODEL_TAG  = "xlmr"   # short tag for filenames; change to "bertic" if MODEL_NAME="classla/bcms-bertic"

# === α = 0.3 — the production setting (Chapter 4.4.5) ===
ALPHA = 0.3

# Load the bilingual PatternReframe splits produced in Phase D.
df_train = pd.read_csv(f"{DATA_DIR}/patternreframe_bilingual_train.csv")
df_val   = pd.read_csv(f"{DATA_DIR}/patternreframe_bilingual_val.csv")
df_test  = pd.read_csv(f"{DATA_DIR}/patternreframe_bilingual_test.csv")

# Drop NaN rows in required columns. `thought_hr` is the Croatian input;
# `pattern` is the English hard label; `marked_*` columns are the annotator
# vote counts used to build soft-label targets.
marked_cols = [c for c in df_train.columns if c.startswith("marked_")]
required_cols = ["thought_hr", "pattern"] + marked_cols
df_train = df_train.dropna(subset=required_cols)
df_val   = df_val.dropna(subset=required_cols)
df_test  = df_test.dropna(subset=["thought_hr", "pattern"])

# Rebuild label dictionaries (sorted alphabetically for determinism — must
# match the English run so that ID indices align across the two notebooks).
classes  = sorted(df_train["pattern"].unique())
label2id = {label: idx for idx, label in enumerate(classes)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"marked_* columns present: {len(marked_cols)}")
print(f"\nLabel mapping (alphabetical):")
for idx, label in id2label.items():
    print(f"  {idx:>2}  {label}")

In [ ]:
# === Cell 5: Tokenise ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df):
    # Use thought_hr (Croatian) as input. Keep the English `pattern` column
    # as the hard label — categorical labels do not translate.
    return Dataset.from_pandas(
        df[["thought_hr", "pattern"]]
            .rename(columns={"thought_hr": "text"})
            .assign(label=lambda d: d["pattern"].map(label2id))
            [["text", "label"]],
        preserve_index=False,
    )

raw_datasets = DatasetDict({
    "train":      make_dataset(df_train),
    "validation": make_dataset(df_val),
    "test":       make_dataset(df_test),
})

# Match the English notebook's MAX_LEN. PR thoughts are short (mean ~25 tokens
# in English; Croatian tokens with XLM-R's sentencepiece will be slightly more
# numerous but still well under 256).
MAX_LEN = 256

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized_datasets = raw_datasets.map(
    tokenize_function, batched=True, remove_columns=["text"]
)

# Sanity check: how many records would hit the cap?
sample_lengths = [len(x["input_ids"]) for x in tokenized_datasets["train"]]
print(f"Token length stats (training set, Croatian):")
print(f"  Mean: {np.mean(sample_lengths):.1f}, Max: {max(sample_lengths)}, Min: {min(sample_lengths)}")
print(f"  Records hitting {MAX_LEN}-token cap: {sum(1 for l in sample_lengths if l == MAX_LEN)}")

In [ ]:
# === Cell 6: Soft-label training (PatternReframe, 10 classes, α = 0.3) ===
# 1. Mapping: marked_* column → label name (exactly as in label2id).
MARKED_COL_TO_PATTERN = {
    "marked_black_and_white":      "Black-and-white or polarized thinking / All or nothing thinking",
    "marked_catastrophizing":      "Catastrophizing",
    "marked_discounting_positive": "Discounting the positive",
    "marked_fortune_telling":      "Jumping to conclusions: Fortune-telling",
    "marked_labeling":             "Labeling and mislabeling",
    "marked_mental_filtering":     "Mental filtering",
    "marked_mind_reading":         "Jumping to conclusions: mind reading",
    "marked_overgeneralization":   "Overgeneralization",
    "marked_personalization":      "Personalization",
    "marked_should_statements":    "Should statements",
}
# Note: marked_emotional_reasoning is excluded — it is always 0 in train.
# Note: marked_none is excluded — our 10-class model has no None output, so
#       any probability mass on it would have nowhere to go.

assert len(MARKED_COL_TO_PATTERN) == len(label2id), \
    f"Mapping has {len(MARKED_COL_TO_PATTERN)} entries but label2id has {len(label2id)}"

# 2. Build soft-label arrays
def build_soft_labels(df, num_labels, alpha=1.0):
    """alpha=1.0 → pure annotator distribution; alpha=0.0 → one-hot on hard label."""
    soft = np.zeros((len(df), num_labels), dtype=np.float32)
    for col, pattern in MARKED_COL_TO_PATTERN.items():
        soft[:, label2id[pattern]] = df[col].values.astype(np.float32)

    row_sums = soft.sum(axis=1)
    no_votes_mask = row_sums == 0
    n_fallback = int(no_votes_mask.sum())
    if n_fallback > 0:
        print(f"  {n_fallback} rows had no votes on the 10 distortion patterns; "
              f"using one-hot fallback on the hard label.")
    for i in np.where(no_votes_mask)[0]:
        soft[i, label2id[df.iloc[i]["pattern"]]] = 1.0
        row_sums[i] = 1.0
    soft = soft / row_sums[:, None]

    # Interpolate with hard one-hot
    if alpha < 1.0:
        hard = np.zeros_like(soft)
        for i, p in enumerate(df["pattern"].values):
            hard[i, label2id[p]] = 1.0
        soft = alpha * soft + (1.0 - alpha) * hard
    return soft

print("Building soft-label arrays (α = {:.2f}):".format(ALPHA))
train_soft = build_soft_labels(df_train, num_labels=len(label2id), alpha=ALPHA)
val_soft   = build_soft_labels(df_val,   num_labels=len(label2id), alpha=ALPHA)

print(f"\nShapes: train {train_soft.shape}, val {val_soft.shape}")
print(f"Row sums (sanity, should be 1.0): {train_soft.sum(axis=1)[:5]}")
print(f"\nExample (row 0):")
print(f"  Hard label: {df_train.iloc[0]['pattern']}")
for label_idx, label_name in id2label.items():
    if train_soft[0, label_idx] > 0:
        print(f"    {train_soft[0, label_idx]:.3f}  {label_name}")

# 3. Attach soft labels to the tokenised datasets
def add_soft_labels(ds, soft_array):
    return ds.add_column("soft_labels", soft_array.tolist())

tokenized_datasets["train"]      = add_soft_labels(tokenized_datasets["train"], train_soft)
tokenized_datasets["validation"] = add_soft_labels(tokenized_datasets["validation"], val_soft)
# Test set keeps only the hard label — evaluation is against ground truth.

# 4. Custom Trainer with soft-label cross-entropy loss
class SoftLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        soft_labels = inputs.pop("soft_labels") if "soft_labels" in inputs else None
        labels = inputs.pop("labels") if "labels" in inputs else None

        outputs = model(**inputs)
        logits = outputs.logits

        if soft_labels is not None:
            log_probs = F.log_softmax(logits, dim=-1)
            loss = -(soft_labels * log_probs).sum(dim=-1).mean()
        elif labels is not None:
            loss = F.cross_entropy(logits, labels)
        else:
            loss = torch.tensor(0.0, device=logits.device)

        return (loss, outputs) if return_outputs else loss

# 5. Custom collator that adds the soft_labels column to the batch
class SoftLabelCollator:
    def __init__(self, tokenizer):
        self.base = DataCollatorWithPadding(tokenizer=tokenizer)
    def __call__(self, features):
        if "soft_labels" in features[0]:
            soft_labels = torch.tensor(
                [f.pop("soft_labels") for f in features],
                dtype=torch.float32,
            )
        else:
            soft_labels = None
        batch = self.base(features)
        if soft_labels is not None:
            batch["soft_labels"] = soft_labels
        return batch

data_collator = SoftLabelCollator(tokenizer)

# 6. Metrics function — unchanged. Evaluation uses hard labels.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy":    accuracy_score(labels, predictions),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
        "f1_macro":    f1_score(labels, predictions, average="macro"),
    }

# 7. Fresh model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label2id), id2label=id2label, label2id=label2id,
)

# 8. Training arguments — match the English run exactly. Only output_dir changes.
RUN_TAG = f"hr_{MODEL_TAG}_softlabels_alpha{int(ALPHA*100)}"
training_args = TrainingArguments(
    output_dir=f"{DATA_DIR}/xlmr_patternreframe_{RUN_TAG}_seed42",
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_strategy="epoch",
    report_to="none",
    seed=42,
    fp16=False,
    bf16=False,
    remove_unused_columns=False,         # keep soft_labels in the dataset
)

# 9. SoftLabelTrainer
trainer = SoftLabelTrainer(
    model=model, args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
# === Cell 7: Train ===
start = time.time()
train_result = trainer.train()
print(f"\nTraining finished in {(time.time() - start)/60:.1f} minutes")

In [ ]:
# === Cell 8: Save final model ===
FINAL_PATH = f"{DATA_DIR}/xlmr_patternreframe_{RUN_TAG}_final"
trainer.save_model(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)
print(f"Model saved to {FINAL_PATH}")

In [ ]:
# === Cell 9: Test set evaluation ===
test_predictions = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(test_predictions.predictions, axis=-1)
y_true = test_predictions.label_ids

print("=== Test set metrics (Croatian PR) ===")
for k, v in test_predictions.metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# === Cell 10: Save predictions and confusion matrix ===
import matplotlib.pyplot as plt
import seaborn as sns

# Save full predictions with class probabilities
predictions_df = pd.DataFrame({
    "y_true":     y_true,
    "y_pred":     y_pred,
    "true_label": [id2label[i] for i in y_true],
    "pred_label": [id2label[i] for i in y_pred],
})
probs = torch.nn.functional.softmax(
    torch.tensor(test_predictions.predictions), dim=-1
).numpy()
for idx in range(len(id2label)):
    predictions_df[f"prob_{idx}"] = probs[:, idx]
predictions_df.to_csv(
    f"{DATA_DIR}/xlmr_patternreframe_{RUN_TAG}_test_predictions.csv",
    index=False,
)
print(f"Saved predictions")

# Short labels for the confusion matrix axes (full names are too long)
short_labels = {
    "Black-and-white or polarized thinking / All or nothing thinking": "All-or-nothing",
    "Catastrophizing":                                                  "Catastrophizing",
    "Discounting the positive":                                         "Discounting positive",
    "Jumping to conclusions: Fortune-telling":                          "Fortune-telling",
    "Jumping to conclusions: mind reading":                             "Mind reading",
    "Labeling and mislabeling":                                         "Labeling",
    "Mental filtering":                                                 "Mental filter",
    "Overgeneralization":                                               "Overgeneralization",
    "Personalization":                                                  "Personalization",
    "Should statements":                                                "Should statements",
}
labels = [id2label[i] for i in range(len(id2label))]
short_axis = [short_labels.get(l, l) for l in labels]

# Row-normalised confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm_normalized,
    annot=True, fmt=".2f", cmap="Blues",
    xticklabels=short_axis, yticklabels=short_axis,
    cbar_kws={"label": "Proportion of true class"},
    ax=ax,
)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title(f"XLM-RoBERTa ({MODEL_TAG}, HR) on translated PatternReframe test set\n"
             f"Row-normalised confusion matrix (α = {ALPHA})")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    f"{DATA_DIR}/xlmr_patternreframe_{RUN_TAG}_confusion_matrix.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print("Saved confusion matrix")

## Cell 11: Side-by-side with the English run

Loads the English α = 0.3 results (if present in `DATA_DIR`) and prints a paired comparison. Goes into Section 4.5 (Comparison of classification results) of the thesis.

In [ ]:
# Look for the English α=0.3 predictions file, if it exists, and compare.
import os
import json as _json

en_preds_path = f"{DATA_DIR}/roberta_patternreframe_softlabels_alpha30_test_predictions.csv"

if not os.path.exists(en_preds_path):
    print(f"English α=0.3 predictions not found at:\n  {en_preds_path}")
    print("Skipping side-by-side comparison. Manual comparison can be done")
    print("against the numbers reported in Chapter 4.4.5 of the thesis draft")
    print("(Acc ≈ 0.6226, F1 weighted ≈ 0.6214, F1 macro ≈ 0.6229).")
else:
    en_preds = pd.read_csv(en_preds_path)

    en_acc      = accuracy_score(en_preds["y_true"], en_preds["y_pred"])
    en_f1_w     = f1_score(en_preds["y_true"], en_preds["y_pred"], average="weighted")
    en_f1_m     = f1_score(en_preds["y_true"], en_preds["y_pred"], average="macro")

    hr_acc      = accuracy_score(y_true, y_pred)
    hr_f1_w     = f1_score(y_true, y_pred, average="weighted")
    hr_f1_m     = f1_score(y_true, y_pred, average="macro")

    print(f"{'Metric':<20} {'EN':>10} {'HR':>10} {'Δ':>10}")
    print("-" * 52)
    print(f"{'Accuracy':<20} {en_acc:>10.4f} {hr_acc:>10.4f} {hr_acc-en_acc:>+10.4f}")
    print(f"{'F1 weighted':<20} {en_f1_w:>10.4f} {hr_f1_w:>10.4f} {hr_f1_w-en_f1_w:>+10.4f}")
    print(f"{'F1 macro':<20} {en_f1_m:>10.4f} {hr_f1_m:>10.4f} {hr_f1_m-en_f1_m:>+10.4f}")

    # Per-category recall comparison — most useful diagnostic.
    print("\nPer-category recall (diagonal of row-normalised confusion matrix):")
    cm_en = confusion_matrix(en_preds["y_true"], en_preds["y_pred"])
    cm_hr = confusion_matrix(y_true, y_pred)
    diag_en = cm_en.diagonal() / cm_en.sum(axis=1)
    diag_hr = cm_hr.diagonal() / cm_hr.sum(axis=1)

    print(f"{'Category':<55} {'EN':>8} {'HR':>8} {'Δ':>8}")
    print("-" * 80)
    for idx in range(len(id2label)):
        cat = id2label[idx]
        en_r = diag_en[idx] if idx < len(diag_en) else float('nan')
        hr_r = diag_hr[idx] if idx < len(diag_hr) else float('nan')
        delta = hr_r - en_r
        print(f"{cat:<55} {en_r:>8.4f} {hr_r:>8.4f} {delta:>+8.4f}")

## Cell 12: Cross-reference with Phase F marker preservation

The Phase F audit reported per-category marker preservation rates on the PR test corpus. If the Croatian classifier's per-category recall correlates with marker preservation, that supports the narrative "categories with cleaner marker preservation classify better in Croatian". If the correlation is weak or absent (which is what the binary cross-eval suggested), the narrative shifts to "category-level transfer is bottlenecked by something other than marker loss, likely domain register".

Either finding is interesting and citable in Chapter 4.5.

In [ ]:
# Try to load the marker preservation report saved by the Phase F notebook.
import os
marker_path = f"{DATA_DIR}/translation_audit/marker_preservation_pr_test.csv"

if not os.path.exists(marker_path):
    print(f"Marker preservation report not found at:\n  {marker_path}")
    print("Skip this cell or copy the Phase F report into DATA_DIR/translation_audit/.")
else:
    marker_df = pd.read_csv(marker_path)
    # Marker categories use short snake_case names; map to full category names.
    marker_to_full = {
        "black_and_white":      "Black-and-white or polarized thinking / All or nothing thinking",
        "catastrophizing":      "Catastrophizing",
        "discounting_positive": "Discounting the positive",
        "fortune_telling":      "Jumping to conclusions: Fortune-telling",
        "labeling":             "Labeling and mislabeling",
        "mental_filtering":     "Mental filtering",
        "mind_reading":         "Jumping to conclusions: mind reading",
        "overgeneralization":   "Overgeneralization",
        "personalization":      "Personalization",
        "should_statements":    "Should statements",
    }
    marker_df["full_category"] = marker_df["category"].map(marker_to_full)

    # HR per-category recall from the test predictions.
    cm_hr = confusion_matrix(y_true, y_pred)
    diag_hr = cm_hr.diagonal() / cm_hr.sum(axis=1)

    pairs = []
    for idx in range(len(id2label)):
        cat_full = id2label[idx]
        match = marker_df[marker_df["full_category"] == cat_full]
        if len(match):
            pairs.append({
                "category":           cat_full,
                "marker_preservation": float(match["preservation_rate"].iloc[0]),
                "hr_recall":          float(diag_hr[idx]),
            })

    pdf = pd.DataFrame(pairs).sort_values("marker_preservation", ascending=False)
    print("Marker preservation vs HR category recall (sorted by marker preservation):")
    print(f"{'Category':<55} {'MarkPres':>10} {'HR Recall':>10}")
    print("-" * 82)
    for _, r in pdf.iterrows():
        print(f"{r['category']:<55} {r['marker_preservation']:>10.3f} {r['hr_recall']:>10.3f}")

    # Spearman correlation
    from scipy.stats import spearmanr, pearsonr
    rho_s, p_s = spearmanr(pdf["marker_preservation"], pdf["hr_recall"])
    rho_p, p_p = pearsonr(pdf["marker_preservation"], pdf["hr_recall"])
    print(f"\nSpearman ρ: {rho_s:+.3f}  (p = {p_s:.3f})")
    print(f"Pearson  r: {rho_p:+.3f}  (p = {p_p:.3f})")
    print()
    print("Interpretation guide:")
    print("  ρ > +0.6  → marker preservation predicts classifier recall (translation matters)")
    print("  ρ ≈ 0     → recall is bottlenecked elsewhere (domain register, category semantics)")
    print("  ρ < -0.3  → counterintuitive — well-preserved markers are HARDER to classify")